# {{title}}
### *{{date}}*
{{description}}

*msmapper* converts detector images into reciprocal space using transformations contained within the NeXus file.

In [ ]:
import os
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from mmg_toolbox import Experiment, data_file_reader
from mmg_toolbox.utils.env_functions import get_processing_directory

MSMAPPER = '/dls_sw/apps/msmapper/1.9/msm/msmapper'


In [ ]:
# Create experiment object - monitors one or more data folders for files
exp = Experiment('{{experiment_dir}}', instrument='{{beamline}}')

scans = exp.scans({{scan_numbers}})

scan = scans[0]
print(scan)

# Get miniumum hkl-step size
Use the first file and compare the produced hkl with the hkl in the file

In [ ]:
processing_directory = get_processing_directory('{{experiment_dir}}')
out_file = os.path.join(processing_directory, f'msmapper_{scan.scan_number()}')

# [optional] Determine detector pixel positions
central_pixel = [
    scan.map.scannables_length() // 2,
    int(scan('pil3_centre_j')),
    int(scan('pil3_centre_i')),
]
print(f"Central pixel: {central_pixel}")
pixel_file = out_file + '_pixel.nxs'
bean = {
    "inputs": [s.filename for s in scans],
    "output": pixel_file,
    "outputMode": "Coords_HKL",
    "pixelIndexes": [[0, 0, 0], [1, 1, 1], [2, 2, 2], central_pixel],
}
bean_file = out_file + '_bean.json'
print(f"\nCreating msmapper bean at: {bean_file}")
json.dump(bean, open(bean_file, 'w'), indent=4)

print('\n\n############### RUNNING MSMAPPER ################')
result = subprocess.run([MSMAPPER, '-bean', bean_file])
result.check_returncode()
print('############## FINISHED MSMAPPER ################\n\n')

# Get output
pixel_map = data_file_reader(pixel_file)
coords = pixel_map('/processed/reciprocal_space/coordinates')
print(coords)

# Compare with expected hkl
expected_hkl = scan('array([h,k,l])')
msmapper_hkl = coords[-1]
print(f"Expected hkl: {expected_hkl}")
print(f"msmapper hkl: {msmapper_hkl}")
print(f"Difference: {np.sum(np.abs(msmapper_hkl - expected_hkl)):.4f}")

## Run MSMapper

In [ ]:
# main msmapper run
output = out_file + '.nxs'
bean = {
        "inputs": [s.filename for s in scans],  # Filename of scan file
        "output": output,  # Output filename - must be in processing directory, or somewhere you can write to
        "splitterName": "gaussian",  # one of the following strings "nearest", "gaussian", "negexp", "inverse"
        "splitterParameter": 2.0,  # splitter's parameter is distance to half-height of the weight function.
        # If you use None or "" then it is treated as "nearest"
        "scaleFactor": 2.0,
        # the oversampling factor for each image; to ensure that are no gaps in between pixels in mapping
        "step": [0.002],
        # a single value or list if 3 values and determines the lengths of each side of the voxels in the volume
        # "start": start,  # location in HKL space of the bottom corner of the array.
        # "shape": shape,  # size of the array to create for reciprocal space volume
        "reduceToNonZero": True  # True/False, if True, attempts to reduce the volume output
    }
bean_file = out_file + '_bean.json'
print(f"\nCreating msmapper bean at: {bean_file}")
json.dump(bean, open(bean_file, 'w'), indent=4)

print('\n\n############### RUNNING MSMAPPER ################')
result = subprocess.run([MSMAPPER, '-bean', bean_file])
result.check_returncode()
print('############## FINISHED MSMAPPER ################\n\n')

## Load MSMapper output
View the NeXus file [here](https://myhdf5.hdfgroup.org/) (drag and drop your file).

In [ ]:
print(f"File created \n'{output}'")
print(f"File size: {os.stat(output).st_size/1e6} MB")

In [ ]:
# Get output
remap = data_file_reader(output)

with remap.map.load_hdf() as hdf:
    # the following are links to the original scan file
    scan_command = remap.map.get_data(hdf, '/entry0/scan_command')  # str
    crystal = remap.map.get_data(hdf, '/entry0/sample/name')  # str
    temp = remap.map.get_data(hdf, '/entry0/instrument/temperature_controller/Tsample')  # float
    # this is the processed data
    haxis = remap.map.get_data(hdf, '/processed/reciprocal_space/h-axis')  # 1D array, length n
    kaxis = remap.map.get_data(hdf, '/processed/reciprocal_space/k-axis')  # 1D array, length m
    laxis = remap.map.get_data(hdf, '/processed/reciprocal_space/l-axis')  # 1D array, length o
    volume = remap.map.get_data(hdf, '/processed/reciprocal_space/volume')  # 3D array, shape [n,m,o]
    # detector parameters
    pixel_size = remap.map.get_data(hdf, '/entry0/instrument/pil3_100k/module/fast_pixel_direction')  # float, mm
    detector_distance = remap.map.get_data(hdf, '/entry0/instrument/pil3_100k/transformations/origin_offset')  # float, mm

print(f"Loaded file: {output} with volume shape: {volume.shape}")

In [ ]:
# average angle subtended by each pixel
solid_angle = pixel_size ** 2 / detector_distance ** 2  # sr
print(f'Each pixel is normalised by the solid angle: {solid_angle: .4g} sr')

volume = volume * solid_angle

## Plot Summed Cuts

In [ ]:
# Plot summed cuts
plt.figure(figsize=(18, 8), dpi=60)
title = f"{output} '{crystal}' {temp:.3g} K\n{scan_command}"
plt.suptitle(title, fontsize=18)

plt.subplot(131)
plt.plot(haxis, volume.sum(axis=1).sum(axis=1))
plt.xlabel('h-axis (r.l.u.)', fontsize=16)
plt.ylabel('sum axes [1,2]', fontsize=16)

plt.subplot(132)
plt.plot(kaxis, volume.sum(axis=0).sum(axis=1))
plt.xlabel('k-axis (r.l.u.)', fontsize=16)
plt.ylabel('sum axes [0,2]', fontsize=16)

plt.subplot(133)
plt.plot(laxis, volume.sum(axis=0).sum(axis=0))
plt.xlabel('l-axis (r.l.u.)', fontsize=16)
plt.ylabel('sum axes [0,1]', fontsize=16)

# Plot summed images
plt.figure(figsize=(18, 8), dpi=60)
title = f"{output}\n{crystal} {temp:.3g} K: {scan_command}"
plt.suptitle(title, fontsize=20)
plt.subplots_adjust(wspace=0.3)

plt.subplot(131)
K, H = np.meshgrid(kaxis, haxis)
plt.pcolormesh(H, K, volume.sum(axis=2), shading='auto')
plt.xlabel('h-axis (r.l.u.)', fontsize=16)
plt.ylabel('k-axis (r.l.u.)', fontsize=16)
plt.axis('image')
#plt.colorbar()

plt.subplot(132)
L, H = np.meshgrid(laxis, haxis)
plt.pcolormesh(H, L, volume.sum(axis=1), shading='auto')
plt.xlabel('h-axis (r.l.u.)', fontsize=16)
plt.ylabel('l-axis (r.l.u.)', fontsize=16)
plt.axis('image')
#plt.colorbar()

plt.subplot(133)
L, K = np.meshgrid(laxis, kaxis)
plt.pcolormesh(K, L, volume.sum(axis=0), shading='auto')
plt.xlabel('k-axis (r.l.u.)', fontsize=16)
plt.ylabel('l-axis (r.l.u.)', fontsize=16)
plt.axis('image')
plt.colorbar()

plt.show()

## Plot Histogram

In [ ]:
cmap_name='Greys'
cut_ratios=(1e-3, 1e-2, 1e-1)

plt.figure(figsize=(18, 8), dpi=60)
title = f"{output} '{crystal}' {temp:.3g} K\n{scan_command}"
plt.suptitle(title, fontsize=18)

cmap = plt.get_cmap(cmap_name)
alphas = np.linspace(0.1, 1, len(cut_ratios))
max_volume = volume.max()

ax = plt.figure().add_subplot()
n, bins, patches = ax.hist(np.log10(volume[volume > 0].flatten()), 100)

for cut, alpha in zip(cut_ratios, alphas):
    logval = np.log10(cut * max_volume)
    ax.axvline(logval, c='k')
    for patch in patches:
        if patch.xy[0] >= logval:
            patch.set_color(cmap(alpha))

ax.set_xlabel('Log$_{10}$ Voxel Intensity')
ax.set_ylabel('N')
ax.set_title(title)


## Plot Voxel Image

In [ ]:
x_axis, y_axis, z_axis = haxis, kaxis, laxis
x_label, y_label, z_label = 'H (r.l.u.)', 'K (r.l.u.)', 'L (r.l.u.)'
yy, xx, zz = np.meshgrid(y_axis, x_axis, z_axis)

fig = plt.figure(figsize=(12, 10), dpi=100)
ax = fig.add_subplot(projection='3d')
for cut, alpha in zip(cut_ratios, alphas):
    filled = volume > (cut * max_volume)
    nfilled = np.sum(filled)
    if nfilled / volume.size > 0.5:
        print(f"voxel cut={cut}, filled = {nfilled}({nfilled / volume.size:.3%}), skipping...")
        continue
    else:
        print(f"voxel cut={cut}, filled = {nfilled}({nfilled / volume.size:.3%})")
    color = cmap(alpha, alpha=alpha)
    # rebin transparent voxels for speed
    if cut < 0.1:
        ax.voxels(
            xx[::2, ::2, ::2],
            yy[::2, ::2, ::2],
            zz[::2, ::2, ::2],
            filled=filled[::2, ::2, ::2][:-1, :-1, :-1],
            facecolors=color
        )
    else:
        ax.voxels(xx, yy, zz, filled[:-1, :-1, :-1], facecolors=color)

ax.set_xlabel(x_label)
ax.set_ylabel(y_label)
ax.set_zlabel(z_label)
ax.set_title(title)

## PyVista 3D volumetric plot
See https://docs.pyvista.org/

In [ ]:
import pyvista as pv

# Example: Create volumetric data
grid = pv.ImageData()
grid.dimensions = volume.shape
grid.spacing = (haxis[1]-haxis[0], kaxis[1]-kaxis[0], laxis[1]-laxis[0])
grid.origin = (haxis[0], kaxis[0], laxis[1])
grid.point_data["values"] = volume.flatten(order='F')

# Create a plotter
plotter = pv.Plotter()
actor = plotter.add_volume(grid, cmap="viridis", opacity="sigmoid", n_colors=3276)

# Add a faint border (bounding box)
plotter.show_bounds(
    color="silver",      # border color
    grid=True,
    show_xaxis=True,    # optional: hide labels/ticks
    show_yaxis=True,
    show_zaxis=True,
    xtitle='h',
    ytitle='k',
    ztitle='l',
)

# Add colorscale sliders
init_min, init_max = actor.mapper.scalar_range
clim = [np.log10(init_min), np.log10(init_max)]
print(f"clim: {clim}")
crange = [0, np.log10(volume.max())]

eps = 1e-12  # small gap to keep min < max

def set_cmin(v):
    # ensure cmin < cmax
    clim[0] = min(float(v), clim[1] - eps)
    actor.mapper.scalar_range = (10**clim[0], 10**clim[1])
    plotter.render()

def set_cmax(v):
    # ensure cmax > cmin
    clim[1] = max(float(v), clim[0] + eps)
    actor.mapper.scalar_range = (10**clim[0], 10**clim[1])
    plotter.render()

# Use the full data range as slider bounds
plotter.add_slider_widget(
    set_cmin,
    rng=crange,
    value=clim[0],
    title="Min color limit",
    pointa=(0.10, 0.90),
    pointb=(0.90, 0.90),
    style='modern',
)

plotter.add_slider_widget(
    set_cmax,
    rng=crange,
    value=clim[1],
    title="Max color limit",
    pointa=(0.10, 0.75),
    pointb=(0.90, 0.75),
    style='modern',
)

plotter.show()